In [2]:
import pandas as pd

# Define file path
file_path = "GitHub Master 250205.xlsm"

# Load the Excel file and check available sheet names
xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names

# Display sheet names
sheet_names


['Definitions',
 'Visualisation Dashboard',
 'Financial Instruments',
 'High Level Dashboard',
 'Financing Baseline',
 'Funding Baseline',
 'Investment Needs',
 'OSeMOSYS (Input)',
 'FFRM (Input)']

In [3]:
import pandas as pd
def load_excel_data(file_path):
    df_definitions_full = pd.read_excel(file_path, sheet_name="Definitions", engine="openpyxl")
    
    # Extract different sections
    df_param_constraints = df_definitions_full.iloc[15:30, 1:3].fillna("").reset_index(drop=True)
    df_param_constraints.columns = ["Name", "Description"]
    
    df_financing_baseline = df_definitions_full.iloc[15:32, 4:6].fillna("").reset_index(drop=True)
    df_financing_baseline.columns = ["Name", "Description"]
    
    df_funding_baseline = df_definitions_full.iloc[15:23, 7:9].fillna("").reset_index(drop=True)
    df_funding_baseline.columns = ["Name", "Description"]
    
    df_scenarios = df_definitions_full.iloc[36:38, 4:6].fillna("").reset_index(drop=True)
    df_scenarios.columns = ["Name", "Description"]
    
    df_currencies = df_definitions_full.iloc[33:39, 1:3].fillna("").reset_index(drop=True)
    df_currencies.columns = ["Code", "Currency"]
    
    df_technologies = df_definitions_full.iloc[14:49, 10:13].fillna("").reset_index(drop=True)
    df_technologies.columns = ["Name", "Description", "Classification"]
    
    return df_param_constraints, df_financing_baseline, df_funding_baseline, df_scenarios, df_currencies, df_technologies
df_param_constraints, df_financing_baseline, df_funding_baseline, df_scenarios, df_currencies, df_technologies = load_excel_data(file_path)
df_technologies.head()


,Name,Description,Classification
0,Name,Description,Classification
1,PWRBIO001,Biomass power plant,Biomass
2,PWRCOA001,Coal power plant,Coal
3,PWRGEO,Geothermal power plant,Geothermal
4,PWROHC001,Light fuel oil power plant,Oil


In [4]:
class BaseDefinition:
    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description
    
    def __repr__(self):
        return f"{self.__class__.__name__}(name='{self.name}', description='{self.description}')"

class ParameterConstraint(BaseDefinition):
    pass

class FinancingBaseline(BaseDefinition):
    pass

class FundingBaseline(BaseDefinition):
    pass

class Scenario(BaseDefinition):
    pass

class Currency:
    def __init__(self, code: str, currency: str):
        self.code = code
        self.currency = currency
    
    def __repr__(self):
        return f"Currency(code='{self.code}', currency='{self.currency}')"

class Technology:
    def __init__(self, name: str, description: str, classification: str):
        self.name = name
        self.description = description
        self.classification = classification
    
    def __repr__(self):
        return f"Technology(name='{self.name}', description='{self.description}', classification='{self.classification}')"

# Creating a list of objects from extracted data
def load_definitions_from_dataframe(df, cls):
    return [cls(row["Name"], row["Description"]) for _, row in df.iterrows()]

def load_currencies_from_dataframe(df):
    return [Currency(row["Code"], row["Currency"]) for _, row in df.iterrows()]

def load_technologies_from_dataframe(df):
    return [Technology(row["Name"], row["Description"], row["Classification"]) for _, row in df.iterrows()]

# Example usage:
constraints = load_definitions_from_dataframe(df_param_constraints, ParameterConstraint)
baselines = load_definitions_from_dataframe(df_financing_baseline, FinancingBaseline)
fundings = load_definitions_from_dataframe(df_funding_baseline, FundingBaseline)
scenarios = load_definitions_from_dataframe(df_scenarios, Scenario)
currencies = load_currencies_from_dataframe(df_currencies)
technologies = load_technologies_from_dataframe(df_technologies)
for constraint in constraints:
    print(constraint)
for baseline in baselines:
    print(baseline)
for funding in fundings:
    print(funding)
for scenario in scenarios:
    print(scenario)
for currency in currencies:
    print(currency)
for technology in technologies:
    print(technology)


ParameterConstraint(name='CurrencyExchangeRate', description='')
ParameterConstraint(name='GDPGrowth', description='The parameter defines annual percentage growth rate of GDP.')
ParameterConstraint(name='AnnualGDP', description='')
ParameterConstraint(name='Elasticity of energy demand', description='')
ParameterConstraint(name='CAGR of real energy price', description='')
ParameterConstraint(name='DiscountRate', description='The discount rate parameter represents the percentage rate used to discount future cash flows to their present value.')
ParameterConstraint(name='AnnualLoanRepayments', description='')
ParameterConstraint(name='BuildTime', description='The build time parameter refers to the duration, in years, required for the construction of a power plant from the project's start to its completion and operational readiness.')
ParameterConstraint(name='Weighting', description='The weighting parameter indicates the relative contribution of four finance categories (DomesticCommercial,

In [5]:
# # Load the specific sheet "Financing Baseline" including formulas
# df_financing_baseline = pd.read_excel(file_path, sheet_name="Financing Baseline", engine="openpyxl", keep_default_na=False)

# # Extract formulas by reading as raw values (formulas are not evaluated, just extracted as strings if present)
# import openpyxl

# # Load workbook using openpyxl (preserves formulas)
# wb = openpyxl.load_workbook(file_path, data_only=False)  # data_only=False keeps formulas instead of evaluated values

# # Select the specific sheet
# ws = wb["Financing Baseline"]

# # Extract formulas
# formulas = {}
# for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
#     for cell in row:
#         if isinstance(cell.value, str) and cell.value.startswith("="):  # Check for Excel formulas
#             formulas[cell.coordinate] = cell.value

# # Display extracted formulas
# formulas



# We now do the Investment Needs Sheet using the input we have load

In [7]:
import pandas as pd
import numpy as np
# import ace_tools as tools  # For displaying the DataFrame

def cal_invest_needs(file_path):
    """Extracts and constructs the Investment Needs table from OSeMOSYS and FFRM sheets."""

    # Load the necessary sheets
    df_osemosys = pd.read_excel(file_path, sheet_name="OSeMOSYS (Input)", engine="openpyxl")
    df_ffrm = pd.read_excel(file_path, sheet_name="FFRM (Input)", engine="openpyxl")
    
    # Load the Investment Needs sheet to get structure
    df_investment_needs = pd.read_excel(file_path, sheet_name="Investment Needs", engine="openpyxl")
    
    # 🔹 1️⃣ Find column mappings for Investment Needs from OSeMOSYS
    investment_needs_data = []
    
    for index, row in df_investment_needs.iterrows():
        tech_code = row["Technology"]  # Technology identifier from Investment Needs

        # Locate matching column in OSeMOSYS (Assuming tech names are in row 10)
        col_mask = df_osemosys.iloc[9, 1:].astype(str).str.contains(str(tech_code), na=False)
        if col_mask.any():
            col_index = np.where(col_mask)[0][0] + 1  # Get the column index

            # Extract relevant investment data (Example: Years 2020-2070)
            investment_data = df_osemosys.iloc[20:66, col_index].values  # Equivalent to Excel's INDEX()

            # Store data with corresponding year
            years = df_osemosys.iloc[20:66, 0].values  # Year column
            investment_needs_data.append(pd.DataFrame({"Year": years, "Technology": tech_code, "Investment": investment_data}))

    # 🔹 2️⃣ Merge all extracted investment needs data into a single DataFrame
    df_final_investment_needs = pd.concat(investment_needs_data, ignore_index=True)

    # 🔹 3️⃣ Incorporate financing data from FFRM if applicable
    df_ffrm_filtered = df_ffrm[df_ffrm["Technology"].isin(df_final_investment_needs["Technology"])]
    df_merged = df_final_investment_needs.merge(df_ffrm_filtered, on=["Year", "Technology"], how="left")

    # Display the final extracted Investment Needs DataFrame
    tools.display_dataframe_to_user(name="Investment Needs Extracted", dataframe=df_merged)

    return df_merged

# Example Usage
# file_path = "your_excel_file.xlsm"
df_investment_needs = cal_invest_needs(file_path)
df_investment_needs


NameError: name 'extract_investment_needs' is not defined